# Topics in W.E.B. Du Bois's *Crisis* writing

### An end-to-end tour of [`turbotopics`](https://github.com/nealcaren/turbotopics)

This tutorial works the **turbotopics** topic-modeling library through a real
corpus: **704 articles from *The Crisis***, the NAACP magazine Du Bois edited
from its founding in 1910 until he left in 1934. The articles span
**1910–1934** (the great bulk are Du Bois's own editorials), so the corpus lets
us ask not only *what* Du Bois wrote about but *how those topics shifted across
his editorship* — from the agitation and World War I of the 1910s, through the
New Negro / Harlem Renaissance 1920s, into the economic-program early 1930s.

The data file [`examples/dubois_crisis.csv`](dubois_crisis.csv) ships alongside
this notebook; its columns are `title, year, decade, volume, issue, author,
subjects, text`.

We move through turbotopics' models in the order you would in real research:

| # | Step | What it shows |
|---|------|---------------|
| 1 | **Preprocess** | tokenize + stoplist, build a pruned `Corpus` |
| 2 | **Phrases** | learn collocations (*jim crow*, *colored people*) first |
| 3 | **LDA** | a plain topic model, scored with coherence + diversity |
| 4 | **STM** | prevalence on decade — which topics rise/fall over time |
| 5 | **DTM** | dynamic topics — trace a word's trajectory across decades |
| 6 | **HDP** | let the model infer how many topics there are |
| 7 | **Utilities** | `summary()`, save/load, `prepare_pyldavis()` |
| 8 | **transform** | held-out inference on brand-new documents |

Everything here is tuned to run in a couple of minutes. Where a knob trades
speed for quality (`K`, iterations, EM sweeps) we use a modest value and note in
the prose how to scale it up for a real analysis.

In [1]:
import csv
import os
import tempfile

import numpy as np

import turbotopics as tt
from turbotopics import Corpus, DTM, HDP, LDA, STM, stm, tokenize

# Resolve paths relative to this notebook (examples/).
HERE = os.path.dirname(os.path.abspath("dubois_tutorial.ipynb"))
CSV_PATH = os.path.join(HERE, "dubois_crisis.csv")
STOP_PATH = os.path.join(HERE, "english-stoplist.txt")

print("turbotopics", tt.__version__)

turbotopics 0.1.0


## 0. Load the corpus

A quick read of the CSV, and a look at how the articles distribute across the
decades and how many carry Du Bois's own byline.

In [2]:
def load_corpus_rows():
    with open(CSV_PATH, newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    for r in rows:
        r["year"] = int(r["year"])
        r["decade"] = int(r["decade"])
    return rows

rows = load_corpus_rows()
years = [r["year"] for r in rows]
dubois = sum(1 for r in rows if "Du Bois" in r["author"])
print(f"Loaded {len(rows)} articles, {min(years)}-{max(years)}.")
print(f"  Du Bois-authored: {dubois}  ({100 * dubois / len(rows):.0f}%)")

by_decade = {}
for r in rows:
    by_decade[r["decade"]] = by_decade.get(r["decade"], 0) + 1
print("  Articles per decade: "
      + ", ".join(f"{d}s={n}" for d, n in sorted(by_decade.items())))

Loaded 704 articles, 1910-1934.
  Du Bois-authored: 679  (96%)
  Articles per decade: 1910s=356, 1920s=276, 1930s=72


## 1. Preprocess

`turbotopics.tokenize` does the lowercase / regex / stoplist / min-length work
the corpus loader uses internally. We keep the per-document token lists around
(`docs`) because several helpers (coherence, phrases, pyLDAvis) want the
tokenized texts.

We add a handful of **corpus-specific stopwords**: *crisis*, *negro*, *colored*
are ubiquitous here (it **is** *The Crisis*, about "the Negro") and would
otherwise dominate every topic; the rest are high-frequency function-ish words
that survive the English stoplist. Pruning these sharpens every model below.

> **Note:** `stopwords` must be a *sequence* (list), not a set.

In [3]:
EXTRA_STOPS = [
    "crisis", "negro", "negroes", "colored", "mr", "mrs", "dr",
    "shall", "men", "man", "upon", "us", "one", "two", "every",
    "may", "must", "said", "yet", "thing", "things", "make", "made",
]

english_stops = open(STOP_PATH, encoding="utf-8").read().split()
stopwords = sorted(set(english_stops) | set(EXTRA_STOPS))
print(f"Stoplist: {len(english_stops)} English + "
      f"{len(EXTRA_STOPS)} corpus-specific = {len(stopwords)} words.")

# min_length=3 drops 1-2 char OCR noise.
docs = [tokenize(r["text"], stopwords=stopwords, min_length=3) for r in rows]
raw_vocab = len({w for d in docs for w in d})
print(f"Tokenized {len(docs)} documents -> {raw_vocab} raw word types.")

# Build a Corpus, pruning the vocabulary tomotopy-style:
#   min_doc_freq=10  -> a word must appear in >=10 documents (kills hapaxes)
#   rm_top=20        -> drop the 20 most frequent words (residual stopwords)
corpus = Corpus.from_documents(docs, min_doc_freq=10, rm_top=20)
print(f"Pruned Corpus: {corpus.num_docs} docs, vocab={corpus.num_words}, "
      f"{corpus.total_tokens} tokens after pruning.")

Stoplist: 140 English + 23 corpus-specific = 155 words.


Tokenized 704 documents -> 21613 raw word types.
Pruned Corpus: 704 docs, vocab=3391, 167709 tokens after pruning.


## 2. Phrases

Du Bois's prose is full of fixed expressions (*jim crow*, *colored people*,
*souls of black folk*). Detecting them **before** modeling means a topic can be
about *the phrase* rather than its parts scattered across topics.
`learn_phrases` scores adjacent word pairs; `apply_phrases` rewrites the token
lists, joining survivors with `_`. We then rebuild the `Corpus` from the phrased
documents — every later model trains on it.

In [4]:
phrase_model = tt.learn_phrases(docs, min_count=8, threshold=12.0)
phrased_docs = tt.apply_phrases(docs, phrase_model)
detected = sorted({w for d in phrased_docs for w in d if "_" in w})
print(f"Detected {len(detected)} multiword phrases. A sample:")

wanted = ["jim_crow", "colored_people", "booker_washington",
          "civil_rights", "abraham_lincoln", "atlanta_university"]
show = [p for p in wanted if p in detected] or detected[:6]
show += [p for p in detected if p not in show][:8]
for p in show[:12]:
    print(f"    {p.replace('_', ' ')}")

pcorpus = Corpus.from_documents(phrased_docs, min_doc_freq=10, rm_top=20)
print(f"\nPhrased Corpus vocab: {pcorpus.num_words}.")

Detected 118 multiword phrases. A sample:
    jim crow
    booker washington
    civil rights
    abraham lincoln
    atlanta university
    anti-lynching bill
    attitude toward
    bad treatment
    board directors
    bread butter
    brown yellow
    civil service

Phrased Corpus vocab: 3418.


## 3. LDA — the workhorse

We fit **K=15** topics with collapsed Gibbs sampling, read each topic off as its
top words, and score the model two ways:

- **`coherence` (c_v)** — do a topic's top words actually co-occur?
- **`topic_diversity`** — do topics avoid recycling the same words?

*Scale up:* K up to ~40 and iterations to ~1000 for a real run.

In [5]:
K = 15
lda = LDA(num_topics=K, seed=1)
lda.fit(pcorpus, iterations=400, num_samples=4, sample_interval=25)
print(f"Fit LDA(K={K}). Top words per topic:")
for t in range(K):
    words = [w for w, _ in lda.top_words(8, topic=t)]
    print(f"  T{t:2d}: {', '.join(w.replace('_', ' ') for w in words)}")

Fit LDA(K=15). Top words per topic:
  T 0: folk, life, truth, new york, human, come, art, freedom
  T 1: schools, education, children, public, segregation, training, college, teachers
  T 2: house, came, went, mob, back, saw, home, street
  T 3: lynching, murder, state, law, crime, case, mob, governor
  T 4: fight, rights, democracy, without, public, nation, long, problem
  T 5: god, face, little, eyes, came, children, dark, long
  T 6: labor, economic, industrial, industry, today, workers, capital, class
  T 7: say, want, know, let, social, good, friends, believe
  T 8: war, africa, races, england, peace, europe, civilization, government
  T 9: votes, voters, party, political, states, election, state, disfranchisement
  T10: property, large, north, population, whites, conditions, number, land
  T11: business, africa, editor, magazine, money, capital, pay, month
  T12: city, part, chicago, hundred, among, three, miss, put
  T13: church, committee, organization, conference, general, mem

In [6]:
coh = tt.coherence(lda, phrased_docs, coherence_type="c_v", topn=10)
div = tt.topic_diversity(lda, topn=15)
print(f"Mean c_v coherence: {coh.mean():.3f}  "
      f"(best topic {coh.argmax()} = {coh.max():.3f}, "
      f"worst topic {coh.argmin()} = {coh.min():.3f})")
print(f"Topic diversity (top-15): {div:.3f}  "
      f"(1.0 = every top word unique to its topic)")

Mean c_v coherence: 0.612  (best topic 14 = 0.864, worst topic 11 = 0.416)
Topic diversity (top-15): 0.884  (1.0 = every top word unique to its topic)


## 4. STM — topic prevalence over time

The **Structural Topic Model** lets topic *prevalence* depend on document
metadata. We make prevalence a function of **decade** (one-hot dummies via
`turbotopics.one_hot`), so the model can express that, say, a Harlem-Renaissance
topic is more common in the 1920s.

To ask *"which topics rose or fell over time"* with proper uncertainty we use
the **method of composition** (Treier & Jackman 2008), exactly as the R `stm`
package does:

1. `posterior_theta_samples` → draws of the document-topic matrix θ,
2. `estimate_effect(draws, X=year)` → regress each topic on year, pooling the
   draws by Rubin's rules so the SEs include topic-estimation uncertainty.

A positive, significant `year` slope means the topic grew over time.

In [7]:
Ks = 12
X_decade, decade_names = tt.one_hot([r["decade"] for r in rows], prefix="dec")
print(f"Prevalence design: {X_decade.shape[1]} decade dummies "
      f"({', '.join(decade_names)}); reference decade dropped.")

stm_model = STM(num_topics=Ks, seed=1)
# em_iters modest for runtime; ~75-100 for a real fit.
stm_model.fit(phrased_docs, X_decade, prevalence_names=decade_names, em_iters=30)
print(f"Fit STM(K={Ks}) with decade prevalence.")

Prevalence design: 2 decade dummies (dec1920, dec1930); reference decade dropped.


Fit STM(K=12) with decade prevalence.


Interpret topics with stm-style word lists: **prob** (most frequent) vs **FREX**
(frequent *and* exclusive to the topic — usually the most evocative).

In [8]:
labels = stm.label_topics(stm_model.topic_word, stm_model.vocabulary, n=7)
print("Topic labels (Highest-probability vs FREX words):")
for t in range(Ks):
    prob = ", ".join(w.replace("_", " ") for w, _ in labels[t]["prob"])
    frex = ", ".join(w.replace("_", " ") for w, _ in labels[t]["frex"])
    print(f"  T{t:2d}  prob: {prob}")
    print(f"       frex: {frex}")

Topic labels (Highest-probability vs FREX words):
  T 0  prob: white, world, black, race, people, truth, american
       frex: dominant wills, van vechten, porgy, novels, toomer, brazil, cabarets
  T 1  prob: white, people, south, black, race, labor, american
       frex: scottsboro, curtailed, rents, capitalists, capitalistic, communists, laboring
  T 2  prob: school, schools, white, children, education, work, public
       frex: curriculum, endowment, fayetteville, talladega, alumni, blair, woofter's
  T 3  prob: congress, world, great, africa, black, war, land
       frex: gandhi, princess, matabele, king yonder, byrnes, natal, thro
  T 4  prob: vote, south, white, black, women, votes, united states
       frex: democrats, republicans, presidential, presidential election, democratic ticket, perry howard, tammany
  T 5  prob: white, officers, french, regiment, camp, division, first
       frex: argonne, pitts, sector, machine gun, field artillery, okolona, brigade
  T 6  prob: mob, w

In [9]:
# Method of composition: regress each topic's proportion on YEAR.
year = np.array([r["year"] for r in rows], dtype=float).reshape(-1, 1)
theta_draws = stm.posterior_theta_samples(stm_model, nsims=20, seed=0)
effects = stm.estimate_effect(theta_draws, year, feature_names=["year"])

trends = []
for t in range(Ks):
    d = effects[t].as_dict()
    trends.append((t, d["year"]["coef"], d["year"]["z"]))

risers = sorted(trends, key=lambda x: x[1], reverse=True)
short = lambda t: ", ".join(w.replace("_", " ") for w, _ in labels[t]["frex"][:3])

print("Topic trends over time (per-year change in prevalence):")
print("  Rising fastest:")
for t, coef, z in risers[:3]:
    star = "*" if abs(z) > 1.96 else " "
    print(f"    T{t:2d} {coef:+.5f}/yr (z={z:+.2f}){star}  [{short(t)}]")
print("  Falling fastest:")
for t, coef, z in risers[-3:][::-1]:
    star = "*" if abs(z) > 1.96 else " "
    print(f"    T{t:2d} {coef:+.5f}/yr (z={z:+.2f}){star}  [{short(t)}]")
print("    (* = |z| > 1.96; year coef is the OLS slope of theta on year.)")

Topic trends over time (per-year change in prevalence):
  Rising fastest:
    T11 +0.00307/yr (z=+3.01)*  [porters, pullman, amenia]
    T 1 +0.00249/yr (z=+1.52)   [scottsboro, curtailed, rents]
    T 2 +0.00157/yr (z=+1.09)   [curriculum, endowment, fayetteville]
  Falling fastest:
    T 3 -0.00312/yr (z=-2.66)*  [gandhi, princess, matabele]
    T 9 -0.00274/yr (z=-2.13)*  [schirmer, vines, pyrenees]
    T 4 -0.00227/yr (z=-1.36)   [democrats, republicans, presidential]
    (* = |z| > 1.96; year coef is the OLS slope of theta on year.)


## 5. DTM — word trajectories across decades

The **Dynamic Topic Model** fixes the number of topics but lets each topic's
word distribution **drift** across ordered time slices. We slice by decade and
trace a salient word's probability through time within the topic where it lives.

`chain_variance` controls how freely a topic may drift between slices. The
default (`0.005`) is deliberately stiff and barely moves on a 3-slice corpus;
`0.05` lets real trends show while staying smooth. The two clearest stories:
**war** (the WWI vocabulary of the 1910s receding) and **labor** (Du Bois's
economic/Marxist language climbing toward the 1930s).

In [10]:
decades = sorted(by_decade)            # contiguous, 0-based time indices
didx = {d: i for i, d in enumerate(decades)}
times = [didx[r["decade"]] for r in rows]

dtm = DTM(num_topics=8, chain_variance=0.05, seed=1)
dtm.fit(pcorpus, times, em_iters=20)   # ~30+ EM iters for a real run
print(f"Fit DTM(K=8) over {dtm.num_times} decade slices "
      f"({decades[0]}s ... {decades[-1]}s).")


def best_topic_for(word):
    # The DTM topic in which `word` is most probable (averaged over time).
    if word not in dtm.vocabulary:
        return None
    per_time = np.stack([dtm.topic_word(s) for s in range(dtm.num_times)])
    wid = list(dtm.vocabulary).index(word)
    return int(per_time[:, :, wid].mean(axis=0).argmax())


print("\nWord probability trajectories (x1000) by decade:")
print("    " + "  ".join(f"{d}s" for d in decades))
# "war" and "labor" tell the clearest story; africa peaks in the Pan-African
# 1920s; lynching stays roughly constant (a perennial theme).
for word in ["war", "labor", "africa", "lynching", "schools"]:
    topic = best_topic_for(word)
    if topic is None:
        print(f"    {word:10s} (not in pruned vocab)")
        continue
    evo = dtm.word_evolution(topic, word)
    cells = "  ".join(f"{1000 * float(p):5.2f}" for p in evo)
    print(f"  {word:10s} (T{topic}): {cells}")

Fit DTM(K=8) over 3 decade slices (1910s ... 1930s).

Word probability trajectories (x1000) by decade:
    1910s  1920s  1930s
  war        (T4):  7.34   6.53   6.28
  labor      (T1): 13.06  15.40  18.55
  africa     (T7): 15.93  21.73  23.87
  lynching   (T6):  4.23   4.95   5.01
  schools    (T0): 16.84  16.25  14.06


## 6. HDP — infer the number of topics

The **Hierarchical Dirichlet Process** is nonparametric: instead of you picking
`K`, it infers how many topics the data support. Handy as a sanity check on the
`K` you chose for LDA/STM above.

`eta` is the topic-word smoothing. The default (`0.01`) is sharp and
over-segments this OCR-noisy corpus into dozens of tiny topics; `0.3` is smoother
and lands on a sensible count. HDP topics aren't ordered by size, so we rank them
by prevalence (total θ mass) and show the largest — the small tail topics are
mostly noise.

In [11]:
hdp = HDP(eta=0.3, seed=1)
hdp.fit(pcorpus, iters=150)            # ~300+ iters for a stable estimate
print(f"HDP inferred K = {hdp.num_topics} topics from the data "
      f"(reassuringly close to the K=15 we picked for LDA, K=12 for STM).")

mass = hdp.doc_topic.sum(axis=0)       # total prevalence of each topic
order = np.argsort(mass)[::-1]
print("The most prevalent inferred topics:")
for t in order[:6]:
    words = [w.replace("_", " ") for w, _ in hdp.top_words(8, topic=int(t))]
    print(f"  ({mass[t]:5.1f} docs) {', '.join(words)}")

HDP inferred K = 17 topics from the data (reassuringly close to the K=15 we picked for LDA, K=12 for STM).
The most prevalent inferred topics:
  (235.1 docs) social, problem, folk, know, today, education, far, without
  ( 88.9 docs) political, votes, voters, states, party, southern, state, disfranchisement
  ( 77.3 docs) little, god, came, face, come, woman, eyes, last
  ( 61.8 docs) mob, lynching, murder, police, state, case, law, crime
  ( 58.2 docs) war, land, slavery, country, today, fight, let, peace
  ( 51.6 docs) schools, city, children, property, public, education, church, new york


## 7. Utilities

Three things you'll reach for constantly:

- **`summary()`** — a tomotopy-style one-shot overview of a fitted model.
- **`save` / `load`** — persist a fitted model and read it straight back.
- **`prepare_pyldavis()`** — builds the intertopic-distance visualization. If
  `pyLDAvis` is installed this returns its `PreparedData` (pass to
  `pyLDAvis.save_html`); otherwise you get a `PyLDAvisInputs` you can feed to
  `pyLDAvis.prepare` later.

In [12]:
print("turbotopics.summary(lda):\n")
print(tt.summary(lda, topn=6))

turbotopics.summary(lda):

LDA(num_topics=15, beta=0.01, fitted=true)
  num_topics: 15
  alpha: [1.08034418 0.61732166 0.58147149 0.58297969 1.36130719 0.66075956
 0.74470003 1.50104511 0.68371274 0.46498839 0.75141045 0.62756163
 0.98181897 0.79039833 0.29215266]
  vocab_size: 3418
  topic 0: folk life truth new_york human come
  topic 1: schools education children public segregation training
  topic 2: house came went mob back saw
  topic 3: lynching murder state law crime case
  topic 4: fight rights democracy without public nation
  topic 5: god face little eyes came children
  topic 6: labor economic industrial industry today workers
  topic 7: say want know let social good
  topic 8: war africa races england peace europe
  topic 9: votes voters party political states election
  topic 10: property large north population whites conditions
  topic 11: business africa editor magazine money capital
  topic 12: city part chicago hundred among three
  topic 13: church committee organiza

In [13]:
model_path = os.path.join(tempfile.gettempdir(), "dubois_lda.bin")
lda.save(model_path)
reloaded = LDA.load(model_path)
print(f"Saved LDA to {model_path} and reloaded it "
      f"(K={reloaded.num_topics}, vocab={len(reloaded.vocabulary)}).")
os.remove(model_path)

vis = stm.prepare_pyldavis(lda, phrased_docs)
print(f"prepare_pyldavis -> {type(vis).__name__} "
      "(install pyLDAvis to render it as an interactive HTML chart).")

Saved LDA to /tmp/claude-501/dubois_lda.bin and reloaded it (K=15, vocab=3418).
prepare_pyldavis -> PyLDAvisInputs (install pyLDAvis to render it as an interactive HTML chart).


## 8. Held-out inference: `transform`

Every model exposes a sklearn-style **`transform`** that infers topic
proportions θ for *new, unseen* documents against the fitted topics — the
variational E-step for CTM/STM, collapsed Gibbs for LDA/DMR/HDP/etc.
Out-of-vocabulary tokens are dropped. Here we hand the fitted LDA two short
synthetic snippets and watch them load on the right topics.

In [14]:
new_docs = [
    "the mob lynched a man in the southern state and the murder went unpunished".split(),
    "children in the public schools need education and college training".split(),
]
theta = lda.transform(new_docs)
print("Inferred topic proportions for 2 held-out snippets:\n")
for i, row in enumerate(theta):
    top = row.argmax()
    words = ", ".join(w for w, _ in lda.top_words(6, topic=int(top)))
    print(f"  snippet {i}: top topic T{top} (p={row[top]:.2f}) -> {words}")

Inferred topic proportions for 2 held-out snippets:

  snippet 0: top topic T3 (p=0.25) -> lynching, murder, state, law, crime, case
  snippet 1: top topic T1 (p=0.39) -> schools, education, children, public, segregation, training


---

That's the full turbotopics workflow on Du Bois's *Crisis*. Scale up `K`,
iterations, and EM sweeps (noted inline) for a publication-grade analysis.